---
## Persiapan Ulang untuk Latihan

Inisialisasi `SparkSession`, konsistensi interpreter driver/worker, dan pembuatan DataFrame `df` dari dataset simulasi 2.000 transaksi e-commerce.


In [1]:
import os
import sys
import numpy as np
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, abs as spark_abs

# Memastikan interpreter Python driver dan worker selaras
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

# Inisialisasi SparkSession
spark = SparkSession.builder \
    .appName("Latihan11") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.shuffle.partitions", "4")

# Menyiapkan dataset simulasi e-commerce
np.random.seed(42)
n = 2000
kategori_list = ["Elektronik", "Fashion", "Makanan", "Rumah Tangga", "Kesehatan"]
kota_list = ["Magelang", "Yogyakarta", "Semarang", "Solo"]

data = pd.DataFrame({
    "unit_terjual": np.random.randint(1, 15, size=n),
    "harga_satuan": np.random.choice([25000, 50000, 75000, 100000, 150000], size=n),
    "kategori": np.random.choice(kategori_list, size=n),
    "kota": np.random.choice(kota_list, size=n),
    "rating": np.random.randint(1, 6, size=n),
})
data["pendapatan"] = data["unit_terjual"] * data["harga_satuan"] + np.random.normal(0, 20000, size=n)

df = spark.createDataFrame(data)
print("Siap untuk latihan. Total baris:", df.count())
df.show(5)

26/11/12 17:51:43 WARN Utils: Your hostname, raka-Aspire-A314-32 resolves to a loopback address: 127.0.1.1; using 10.90.70.61 instead (on interface wlp3s0)
26/11/12 17:51:43 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/11/12 17:51:44 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Siap untuk latihan. Total baris: 2000


+------------+------------+------------+----------+------+------------------+
|unit_terjual|harga_satuan|    kategori|      kota|rating|        pendapatan|
+------------+------------+------------+----------+------+------------------+
|           7|      150000|     Fashion|  Magelang|     4| 997913.0612957058|
|           4|      100000|Rumah Tangga|  Semarang|     4| 389490.8317158621|
|          13|      150000|  Elektronik|Yogyakarta|     4|1936047.6709878189|
|          11|      100000|Rumah Tangga|  Magelang|     5|1080918.9109179082|
|           8|      150000|     Makanan|  Semarang|     2|1191696.4943767258|
+------------+------------+------------+----------+------+------------------+
only showing top 5 rows



---
### Soal 1
> Ulangi proses `StringIndexer` untuk kolom `kategori` dan `kota`, lalu buat `VectorAssembler` **baru** yang HANYA menggunakan tiga fitur: `unit_terjual`, `harga_satuan`, dan `rating` (tanpa `kategori_idx`/`kota_idx`). Beri nama kolom hasil `features_sederhana`.


In [2]:
# Jawaban Soal 1
from pyspark.ml.feature import StringIndexer, VectorAssembler

# 1. Mengulangi proses StringIndexer untuk kategori dan kota
indexer_kategori = StringIndexer(inputCol="kategori", outputCol="kategori_idx")
indexer_kota = StringIndexer(inputCol="kota", outputCol="kota_idx")

df_indexed = indexer_kategori.fit(df).transform(df)
df_indexed = indexer_kota.fit(df_indexed).transform(df_indexed)

# 2. Membuat VectorAssembler baru HANYA menggunakan 3 fitur: unit_terjual, harga_satuan, dan rating
assembler_sederhana = VectorAssembler(
    inputCols=["unit_terjual", "harga_satuan", "rating"],
    outputCol="features_sederhana"
)

# Menerapkan assembler ke DataFrame
df_sederhana = assembler_sederhana.transform(df_indexed)

print("Skema DataFrame dengan features_sederhana:")
df_sederhana.select("unit_terjual", "harga_satuan", "rating", "features_sederhana", "pendapatan").show(5, truncate=False)

Skema DataFrame dengan features_sederhana:


+------------+------------+------+-------------------+------------------+
|unit_terjual|harga_satuan|rating|features_sederhana |pendapatan        |
+------------+------------+------+-------------------+------------------+
|7           |150000      |4     |[7.0,150000.0,4.0] |997913.0612957058 |
|4           |100000      |4     |[4.0,100000.0,4.0] |389490.8317158621 |
|13          |150000      |4     |[13.0,150000.0,4.0]|1936047.6709878189|
|11          |100000      |5     |[11.0,100000.0,5.0]|1080918.9109179082|
|8           |150000      |2     |[8.0,150000.0,2.0] |1191696.4943767258|
+------------+------------+------+-------------------+------------------+
only showing top 5 rows



---
### Soal 2
> Latih model `LinearRegression` baru menggunakan `features_sederhana` dari Soal 1 (gunakan train/test split yang sama, `seed=42`). Bandingkan nilai **R²** model ini dengan model utama pada Sub-bab 11.4 yang menggunakan 5 fitur — apakah performanya lebih baik, sama, atau lebih buruk?


In [3]:
# Jawaban Soal 2
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

# 1. Train/Test split dengan seed=42 (80% training, 20% testing)
train_sederhana, test_sederhana = df_sederhana.randomSplit([0.8, 0.2], seed=42)

# 2. Melatih model LinearRegression baru menggunakan features_sederhana
lr_sederhana = LinearRegression(featuresCol="features_sederhana", labelCol="pendapatan")
model_sederhana = lr_sederhana.fit(train_sederhana)

# 3. Prediksi pada data testing
pred_sederhana = model_sederhana.transform(test_sederhana)

# 4. Evaluasi performa model sederhana (R² dan RMSE)
evaluator_r2 = RegressionEvaluator(labelCol="pendapatan", predictionCol="prediction", metricName="r2")
evaluator_rmse = RegressionEvaluator(labelCol="pendapatan", predictionCol="prediction", metricName="rmse")

r2_sederhana = evaluator_r2.evaluate(pred_sederhana)
rmse_sederhana = evaluator_rmse.evaluate(pred_sederhana)

# Menghitung juga model utama 5 fitur pada sesi saat ini untuk perbandingan adil
assembler_5 = VectorAssembler(
    inputCols=["unit_terjual", "harga_satuan", "kategori_idx", "kota_idx", "rating"],
    outputCol="features"
)
df_5 = assembler_5.transform(df_indexed)
train_5, test_5 = df_5.randomSplit([0.8, 0.2], seed=42)
lr_5 = LinearRegression(featuresCol="features", labelCol="pendapatan")
model_5 = lr_5.fit(train_5)
pred_5 = model_5.transform(test_5)
r2_5 = evaluator_r2.evaluate(pred_5)
rmse_5 = evaluator_rmse.evaluate(pred_5)

print("=== HASIL EVALUASI MODEL SEDERHANA (3 FITUR) ===")
print(f"Koefisien [unit_terjual, harga_satuan, rating]: {model_sederhana.coefficients}")
print(f"Intercept: {model_sederhana.intercept:,.2f}")
print(f"RMSE : Rp {rmse_sederhana:,.2f}")
print(f"R²   : {r2_sederhana:.4f} ({r2_sederhana*100:.2f}%)")

print("\n=== PERBANDINGAN DENGAN MODEL UTAMA (5 FITUR) ===")
print(f"Model Utama (5 fitur)     -> R²: {r2_5:.4f} | RMSE: Rp {rmse_5:,.2f}")
print(f"Model Sederhana (3 fitur) -> R²: {r2_sederhana:.4f} | RMSE: Rp {rmse_sederhana:,.2f}")
print(f"Selisih R² (Sederhana - Utama): {r2_sederhana - r2_5:+.6f}")

print("\n=== ANALISIS PERBANDINGAN ===")
print("Performa model sederhana (3 fitur) hampir SAMA PERSIS dengan model utama (5 fitur),")
print(f"dengan nilai R² yang sedikit lebih tinggi ({r2_sederhana:.4f} vs {r2_5:.4f}) dan RMSE yang sedikit lebih rendah.")
print("Penyebabnya: Target 'pendapatan' pada data simulasi dibuat dari formula (unit_terjual x harga_satuan) + noise.")
print("Variabel 'kategori' dan 'kota' tidak memiliki korelasi kausal sejati dengan pendapatan,")
print("sehingga menghapusnya tidak mengurangi akurasi, melainkan justru mereduksi sedikit variasi noise (overfitting).")

=== HASIL EVALUASI MODEL SEDERHANA (3 FITUR) ===
Koefisien [unit_terjual, harga_satuan, rating]: [80482.79337026912,7.60959888041514,-4748.6830000562695]
Intercept: -594,521.21
RMSE : Rp 158,965.28
R²   : 0.8961 (89.61%)

=== PERBANDINGAN DENGAN MODEL UTAMA (5 FITUR) ===
Model Utama (5 fitur)     -> R²: 0.8955 | RMSE: Rp 159,448.26
Model Sederhana (3 fitur) -> R²: 0.8961 | RMSE: Rp 158,965.28
Selisih R² (Sederhana - Utama): +0.000632

=== ANALISIS PERBANDINGAN ===
Performa model sederhana (3 fitur) hampir SAMA PERSIS dengan model utama (5 fitur),
dengan nilai R² yang sedikit lebih tinggi (0.8961 vs 0.8955) dan RMSE yang sedikit lebih rendah.
Penyebabnya: Target 'pendapatan' pada data simulasi dibuat dari formula (unit_terjual x harga_satuan) + noise.
Variabel 'kategori' dan 'kota' tidak memiliki korelasi kausal sejati dengan pendapatan,
sehingga menghapusnya tidak mengurangi akurasi, melainkan justru mereduksi sedikit variasi noise (overfitting).


---
### Soal 3
> Menggunakan model utama (5 fitur) dari Sub-bab 11.4, tampilkan **5 baris prediksi dengan selisih (`pendapatan - prediction`) terbesar** — yaitu prediksi yang paling meleset. (Petunjuk: tambahkan kolom baru berisi nilai absolut selisihnya, lalu urutkan.)


In [4]:
# Jawaban Soal 3
# Menggunakan model utama (5 fitur) dan prediksinya pada data test
pred_eval = pred_5.withColumn("selisih", col("pendapatan") - col("prediction")) \
                  .withColumn("abs_selisih", spark_abs(col("selisih"))) \
                  .orderBy(col("abs_selisih").desc())

print("=== 5 BARIS PREDIKSI DENGAN SELISIH TERBESAR (PALING MELESET) PADA MODEL UTAMA ===")
pred_eval.select(
    "unit_terjual", "harga_satuan", "kategori", "kota", "rating",
    "pendapatan", "prediction", "selisih", "abs_selisih"
).show(5, truncate=False)

=== 5 BARIS PREDIKSI DENGAN SELISIH TERBESAR (PALING MELESET) PADA MODEL UTAMA ===


+------------+------------+------------+----------+------+------------------+------------------+-------------------+------------------+
|unit_terjual|harga_satuan|kategori    |kota      |rating|pendapatan        |prediction        |selisih            |abs_selisih       |
+------------+------------+------------+----------+------+------------------+------------------+-------------------+------------------+
|14          |150000      |Rumah Tangga|Yogyakarta|2     |2128292.853349907 |1653824.9558420165|474467.89750789036 |474467.89750789036|
|1           |150000      |Elektronik  |Solo      |1     |171411.58935156555|634375.243897734  |-462963.6545461684 |462963.6545461684 |
|14          |150000      |Makanan     |Magelang  |1     |2120244.488469077 |1661147.6786487347|459096.8098203421  |459096.8098203421 |
|1           |150000      |Fashion     |Yogyakarta|4     |150014.4181387195 |606794.5140171472 |-456780.09587842773|456780.09587842773|
|14          |150000      |Makanan     |Magelang

---
### Soal 4 (Refleksi Singkat)
> Dalam 2-3 kalimat: mengapa Spark MLlib mewajibkan kita menggabungkan seluruh fitur menjadi satu kolom `Vector` (melalui `VectorAssembler`), bukan menerima banyak kolom terpisah seperti kebiasaan di pandas/scikit-learn? Tulis jawaban pada markdown cell di bawah ini.

**Jawaban Refleksi:**
Spark MLlib mewajibkan seluruh fitur digabungkan menjadi satu kolom bertipe `Vector` karena Spark beroperasi dalam arsitektur komputasi terdistribusi di mana algoritma machine learning dioptimalkan menggunakan pustaka aljabar linier tingkat rendah (seperti BLAS dan Breeze) yang memproses data berbasis baris sebagai operasi matriks dan vektor secara seragam dan paralel tanpa *overhead* memetakan puluhan kolom skema terpisah. Selain itu, representasi satu kolom `Vector` secara elegan mendukung format vektor padat (*DenseVector*) maupun vektor jarang (*SparseVector*) yang sangat hemat memori untuk data berdimensi tinggi pada skala Big Data.


---
## Menutup SparkSession


In [5]:
# Menutup SparkSession setelah selesai latihan
spark.stop()
print("SparkSession untuk Latihan Mandiri berhasil ditutup secara bersih dan aman.")

SparkSession untuk Latihan Mandiri berhasil ditutup secara bersih dan aman.
